# OrdinalEncoder — ifri_mini_ml_lib

In data science, most machine learning algorithms—such as linear regression, support vector machines, and neural networks—require input data to be strictly numerical. However, real-world datasets frequently contain categorical variables (text-based or qualitative data like education levels, risk tiers, or geographical regions). Categorical encoding is the process of converting these categorical variables into integer arrays so that mathematical models can process them.

Among the various encoding techniques available (such as One-Hot Encoding or Frequency Encoding), we focus here on Ordinal Encoding.

## 1 · Key concepts
Ordinal Encoding is a specific categorical transformation technique where each unique category in a feature column is mapped to a distinct integer value ranging from $0$ to $n_{\text{categories}} - 1$.

Unlike general Label Encoding, which assigns arbitrary integers to text values, Ordinal Encoding is inherently tied to the assumption of order. It assumes that the categories possess a natural, meaningful hierarchy or scale.

<img src="../../assets/imgs/encoding/ordinal_encoding_representation.jpg" width="1000">

### The Problem of Order Strategy

When dealing with ordinal variables, we typically use two approaches to build our mapping dictionaries:
- **Automated Mode** ('auto'): The encoder automatically extracts unique categories from the training data and sorts them alphabetically. This is a quick, reproducible approach for general variables but can destroy actual physical hierarchies.
- **Manual Mode** (User-Defined): The user explicitly provides ordered lists of categories (e.g., ['Small', 'Medium', 'Large'] or ['S', 'M', 'L']). This ensures that the assigned integers ($0, 1, 2$) perfectly preserve the logical progression of the underlying data.


Unlike parametric models that dynamically update parameters based on an objective optimization function, an encoder is a deterministic preprocessing tool. During the fitting phase, it creates static lookup tables (hash maps) based on the training data categories. During the transformation phase, it uses these stored mappings to encode new, unseen inputs.


## 2. Mathematical Handling & Unknown Values

The mathematical transformation for an individual element $x_{ij}$ (representing the sample row $i$ in the feature column $j$) can be defined as follows:

$$f(x_{ij}) = \text{index of } x_{ij} \text{ within the ordered array of categories } \mathcal{C}_j$$
Where $\mathcal{C}_j$ represents the unique set of categories mapped for column $j$ during the training process.

### Managing Out-Of-Distribution (OOD) Data

One of the most critical challenges in production preprocessing pipelines is encountering an unknown category during inference (a value that was completely missing during the fit phase).Our algorithm provides two strategic guardrails to handle this anomaly:
1. **Strict Enforcement** (handle_unknown='error'): The encoder immediately halts execution and throws a ValueError. This is ideal when your pipeline demands absolute structural data integrity.
2. **Fallback Value** (handle_unknown='use_encoded_value'): The unknown category is gracefully absorbed and mapped to a pre-defined constant integer, typically -1.During an inverse_transform request, any encoded integer that falls outside the valid boundaries of our known category spectrum ($\ge 0$ and $< |\mathcal{C}_j|$) cannot be safely reverse-mapped and is automatically returned as None.

## 3. Pseudo code

1. $\mathcal{D} \leftarrow$ training dataset of $n$ instances and $m$ categorical features ($\mathbf{x}_i \in \mathbb{R}^m$)
2. $\mathbf{x} \leftarrow$ a new dataset instance to transform
3. $\text{mode\_cat} \leftarrow$ strategy for category discovery ($\in \{\text{'auto'}, \text{'user\_defined'}\}$)
4. $\text{unknown\_strat} \leftarrow$ strategy for unseen categories ($\in \{\text{'error'}, \text{'use\_encoded\_value'}\}$)
5. $v_{\text{unknown}} \leftarrow$ default integer replacement value for unseen elements (typically $-1$)
6. $\mathcal{C} \leftarrow$ global list storing unique valid categories for each feature (initially empty)

### 3.1 The Fitting Phase
7. **procedure** $\text{Fit}(\mathcal{D}, \text{mode\_cat})$
8. &nbsp;&nbsp;&nbsp;&nbsp;**for each** feature column $j$ from $1$ to $m$ **do**
9. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**if** $\text{mode\_cat} = \text{'auto'}$ **then**
10. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$\mathcal{C} \leftarrow \text{sort\_alphabetically}(\text{unique\_values}(\mathcal{D}[:, j]))$
11. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**else**
12. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$\mathcal{C} \leftarrow \text{user\_defined\_list}[j]$
13. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**end if**
14. &nbsp;&nbsp;&nbsp;&nbsp;**end for**
15. &nbsp;&nbsp;&nbsp;&nbsp;**return** $\mathcal{C}$
16. **end procedure**

### 3.2 The Transformation Phase
17. **procedure** $\text{Transform}(\mathbf{x}, \mathcal{C}, \text{unknown\_strat}, v_{\text{unknown}})$
18. &nbsp;&nbsp;&nbsp;&nbsp;$\mathbf{x}_{\text{out}} \leftarrow$ empty matrix of the same dimensions as $\mathbf{x}$
19. &nbsp;&nbsp;&nbsp;&nbsp;**for each** feature column $j$ from $1$ to $m$ **do**
20. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$\mathcal{M}_j \leftarrow \text{hash\_map}(\{\text{value} : \text{index} \mid \text{index}, \text{value} \in \text{enumerate}(\mathcal{C}_j)\})$
21. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**for each** row instance $i$ in column $j$ **do**
22. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$val \leftarrow \mathbf{x}[i, j]$
23. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**if** $\text{mode\_cat} = \text{'auto'}$ **then** $val \leftarrow \text{string}(val)$ **end if**
24. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**if** $val \in \mathcal{M}_j$ **then**
25. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$\mathbf{x}_{\text{out}}[i, j] \leftarrow \mathcal{M}_j[val]$
26. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**else if** $\text{unknown\_strat} = \text{'error'}$ **then**
27. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**raise** $\text{ValueError("Unknown category found")}$
28. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**else**
29. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$\mathbf{x}_{\text{out}}[i, j] \leftarrow v_{\text{unknown}}$
30. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**end if**
31. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**end for**
32. &nbsp;&nbsp;&nbsp;&nbsp;**end for**
33. &nbsp;&nbsp;&nbsp;&nbsp;**return** $\mathbf{x}_{\text{out}}$
34. **end procedure**

### 3.3 The Fit-Transform Phase
35. **procedure** $\text{Fit\_Transform}(\mathcal{D}, \text{mode\_cat}, \text{unknown\_strat}, v_{\text{unknown}})$
36. &nbsp;&nbsp;&nbsp;&nbsp; $\mathcal{C} \leftarrow \text{Fit}(\mathcal{D}, \text{mode\_cat})$
37. &nbsp;&nbsp;&nbsp;&nbsp; $\mathbf{x}_{\text{out}} \leftarrow \text{Transform}(\mathcal{D}, \mathcal{C}, \text{unknown\_strat}, v_{\text{unknown}})$
38. &nbsp;&nbsp;&nbsp;&nbsp; **return** $\mathbf{x}_{\text{out}}$
39. **end procedure**

### 3.4 The Inverse Transformation Phase
40. **procedure** $\text{Inverse\_Transform}(\mathbf{x}_{\text{encoded}}, \mathcal{C})$
41. &nbsp;&nbsp;&nbsp;&nbsp; $\mathbf{x}_{\text{inv}} \leftarrow$ empty matrix of the same dimensions as $\mathbf{x}_{\text{encoded}}$
42. &nbsp;&nbsp;&nbsp;&nbsp; **for each** feature column $j$ from $1$ to $m$ **do**
43. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $\text{cats} \leftarrow \mathcal{C}_j$
44. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; **for each** row instance $i$ in column $j$ **do**
45. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $idx \leftarrow \text{integer}(\mathbf{x}_{\text{encoded}}[i, j])$
46. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; **if** $idx \ge 0$ **and** $idx < \text{length}(\text{cats})$ **then**
47. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $\mathbf{x}_{\text{inv}}[i, j] \leftarrow \text{cats}[idx]$
48. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; **else**
49. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $\mathbf{x}_{\text{inv}}[i, j] \leftarrow \text{None}$
50. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; **end if**
51. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; **end for**
52. &nbsp;&nbsp;&nbsp;&nbsp; **end for**
53. &nbsp;&nbsp;&nbsp;&nbsp; **return** $\mathbf{x}_{\text{inv}}$
54. **end procedure**

## 4 · Implementation

To ensure a realistic data science scenario, we synthesize a dedicated customer-profile dataset. This dataset naturally includes multiple ordered hierarchical features: job seniority levels, educational degrees, and corporate scales.

In [13]:
import numpy as np
import pandas as pd
import time
from sklearn.preprocessing import OrdinalEncoder as SklearnOrdinalEncoder

# Seed definition for reproducible benchmarks
np.random.seed(42)
n_samples = 250

# Generating naturally ordered categorical variables
data = {
    "Seniority_Level": np.random.choice(["Junior", "Mid-Level", "Senior", "Director"], size=n_samples, p=[0.4, 0.3, 0.2, 0.1]),
    "Education_Degree": np.random.choice(["High School", "Bachelor", "Master", "PhD"], size=n_samples, p=[0.2, 0.5, 0.2, 0.1]),
    "Company_Size": np.random.choice(["Startup", "SME", "Enterprise"], size=n_samples, p=[0.3, 0.5, 0.2])
}

# Creating the structural NumPy matrix for encoding pipelines
df = pd.DataFrame(data)
X_train = df.values

print(f"Dataset structural dimensions: {X_train.shape}")
print("\nFirst 5 raw observations within the training matrix:")
print(pd.DataFrame(X_train, columns=data.keys()).head())

Dataset structural dimensions: (250, 3)

First 5 raw observations within the training matrix:
  Seniority_Level Education_Degree Company_Size
0          Junior         Bachelor          SME
1        Director         Bachelor          SME
2          Senior           Master          SME
3       Mid-Level         Bachelor   Enterprise
4          Junior      High School          SME


### With Scikit-learn

In [14]:
# Defining explicit hierarchical scales for accurate categorical mapping
custom_categories = [
    ["Junior", "Mid-Level", "Senior", "Director"],  # Feature Column 0
    ["High School", "Bachelor", "Master", "PhD"],    # Feature Column 1
    ["Startup", "SME", "Enterprise"]                 # Feature Column 2
]

# Measuring setup and transformation processing time
start_sklearn = time.perf_counter()

sklearn_encoder = SklearnOrdinalEncoder(
    categories=custom_categories,
    handle_unknown='use_encoded_value',
    unknown_value=-1
)
X_train_encoded_sk = sklearn_encoder.fit_transform(X_train)

duration_sklearn = time.perf_counter() - start_sklearn
print(f"Scikit-Learn total execution wall-time: {duration_sklearn:.6f} seconds")
print("\nEncoded matrix segment (First 5 rows):\n", X_train_encoded_sk[:5])

Scikit-Learn total execution wall-time: 0.001358 seconds

Encoded matrix segment (First 5 rows):
 [[0. 1. 1.]
 [3. 1. 1.]
 [2. 2. 1.]
 [1. 1. 2.]
 [0. 0. 1.]]


### With ifri_mini_ml_lib

In [15]:
# Import enabled via modular editable deployment (pip install -e .)
from ifri_mini_ml_lib.preprocessing.preparation.encoding import OrdinalEncoder

start_ifri = time.perf_counter()

# Replicating structural configuration constraints
ifri_encoder = OrdinalEncoder(
    categories=custom_categories,
    handle_unknown='use_encoded_value',
    unknown_value=-1
)
X_train_encoded_ifri = ifri_encoder.fit_transform(X_train)

duration_ifri = time.perf_counter() - start_ifri
print(f"ifri_mini_ml_lib total execution wall-time: {duration_ifri:.6f} seconds")
print("\nEncoded matrix segment (First 5 rows):\n", X_train_encoded_ifri[:5])

ifri_mini_ml_lib total execution wall-time: 0.000516 seconds

Encoded matrix segment (First 5 rows):
 [[0. 1. 1.]
 [3. 1. 1.]
 [2. 2. 1.]
 [1. 1. 2.]
 [0. 0. 1.]]


In [16]:
# Ensuring element-wise algorithmic correctness via strict NumPy assertions
match_perfect = np.array_equal(X_train_encoded_sk, X_train_encoded_ifri)
print(f"Do encoded outputs mathematically match Scikit-Learn? {match_perfect}")

# Generating empirical summary table for documentation profiles
summary_df = pd.DataFrame({
    'Framework Engine': ['Scikit-Learn (Reference)', 'ifri_mini_ml_lib (Custom)'],
    'Execution Time (s)': [duration_sklearn, duration_ifri],
    'Numerical Integrity': ['Reference Standard', 'PASSED ✅' if match_perfect else 'FAILED ❌']
})
display(summary_df)

Do encoded outputs mathematically match Scikit-Learn? True


,Framework Engine,Execution Time (s),Numerical Integrity
0,Scikit-Learn (Reference),0.001358,Reference Standard
1,ifri_mini_ml_lib (Custom),0.000516,PASSED ✅


#### Case with: categories='auto'
In the following scenario, we switch the configuration to `categories='auto'`. Instead of enforcing a business logic, the encoders must autonomously parse columns, extract unique variables, and establish a mapping sorted alphabetically.

In [17]:
# --- 1. Scikit-Learn Auto Fit ---
sk_auto = SklearnOrdinalEncoder(categories='auto')
X_auto_sk = sk_auto.fit_transform(X_train)

# --- 2. Custom Library Auto Fit ---
ifri_auto = OrdinalEncoder(categories='auto')
X_auto_ifri = ifri_auto.fit_transform(X_train)

# --- 3. Verification ---
auto_match = np.array_equal(X_auto_sk, X_auto_ifri)
print(f"Do automated 'auto' outputs match exactly? {auto_match}")
print("\nDiscovered categories by custom encoder (Column 0 - Alphabetical):")
print(ifri_auto.categories_[0])

Do automated 'auto' outputs match exactly? True

Discovered categories by custom encoder (Column 0 - Alphabetical):
['Director' 'Junior' 'Mid-Level' 'Senior']


#### HANDLING UNSEEN INFERENCE CATEGORIES
A robust machine learning pipeline must gracefully process anomalies during production inference. We feed the fitted custom encoder with a new unseen test matrix containing out-of-distribution values (e.g., `'Intern'`, `'PostDoc'`, `'NGO'`).

In [18]:
# Creating a testing matrix with mixed valid and completely unknown inputs
X_unseen_test = np.array([
    ["Senior", "Master", "SME"],        # 100% Known profile
    ["Intern", "Bachelor", "Startup"],  # 'Intern' is unseen in Column 0
    ["Director", "PostDoc", "NGO"]      # 'PostDoc' (Col 1) and 'NGO' (Col 2) are unseen
])

print("Inference Raw Input Matrix:\n", X_unseen_test)

# 1. Applying Forward Transformation
X_encoded_test = ifri_encoder.transform(X_unseen_test)
print("\nForward Mapping Result (Unseen attributes must evaluate to -1.0):\n", X_encoded_test)

Inference Raw Input Matrix:
 [['Senior' 'Master' 'SME']
 ['Intern' 'Bachelor' 'Startup']
 ['Director' 'PostDoc' 'NGO']]

Forward Mapping Result (Unseen attributes must evaluate to -1.0):
 [[ 2.  2.  1.]
 [-1.  1.  0.]
 [ 3. -1. -1.]]


#### INVERSE MAPPING BACK TO STRINGS
To ensure our data pipeline can accurately translate numerical model predictions back to human-readable insights, we check the behavior of `.inverse_transform()`. Any encoded fallback index (like `-1.0`) must resolve safely to a Python `None` type object.

In [19]:
# Reconstructing original categories from the numeric array
X_recovered = ifri_encoder.inverse_transform(X_encoded_test)

print("Recovered Categorical Array (Out-of-bounds indices must convert to None):\n")
print(X_recovered)

Recovered Categorical Array (Out-of-bounds indices must convert to None):

[['Senior' 'Master' 'SME']
 [None 'Bachelor' 'Startup']
 ['Director' None None]]


## 5. Interactive demo

To explore the behavior of our **OrdinalEncoder dynamically**, this interactive dashboard simulates a real-time production inference pipeline. 

### Interactive Parameters:
1. **Seniority, Education, Company Size**: Dropdown menus to construct a brand-new, unseen observation vector. We included invalid OOD parameters (like **Intern** or **NGO**) to test system resilience.
2. **Handle Unknown Strategy**: Toggle between strict rule enforcement (**error**) and fallback replacement (**use_encoded_value**).
3. **Unknown Value Fill**: Customize the arbitrary integer code used when replacing out-of-distribution categories.

In [1]:
from ipywidgets import interact
from utils import run_interactive_pipeline, seniority_widget, education_widget, company_widget, strategy_widget, fill_value_widget


interact(
    run_interactive_pipeline, 
    seniority=seniority_widget, 
    education=education_widget, 
    company=company_widget, 
    strategy=strategy_widget, 
    fill_value=fill_value_widget
);

interactive(children=(Dropdown(description='Seniority:', options=('Junior', 'Mid-Level', 'Senior', 'Director',…

## 6. Real-life applications

Ordinal encoding is used in machine learning when a categorical variable has a natural, meaningful order or rank, but the intervals between the ranks are not mathematically measurable. [1, 2, 3, 4, 5] 
### Here are the primary real-world applications across various industries:
 **Corporate & HR Systems**

* Job Seniority Levels: Encoding roles from Intern ➔ Junior ➔ Mid-Level ➔ Senior ➔ Lead ➔ Executive to predict employee attrition or expected salary.
* Performance Reviews: Mapping employee evaluations (Needs Improvement ➔ Meets Expectations ➔ Exceeds Expectations) to forecast promotions.
* Education Levels: Ranking candidate qualifications (High School ➔ Bachelor's ➔ Master's ➔ PhD) for automated resume screening tools.


 **E-Commerce & Customer Feedback**

* Customer Loyalty Tiers: Processing user benefits based on status (Bronze ➔ Silver ➔ Gold ➔ Platinum) to predict customer lifetime value (CLV).
* Survey Responses: Analyzing Likert scales (Strongly Disagree ➔ Disagree ➔ Neutral ➔ Agree ➔ Strongly Agree) to train sentiment analysis models.
* Product Sizing: Feeding apparel dimensions (XS ➔ S ➔ M ➔ L ➔ XL) into recommendation engines to suggest the correct fit.

 **Healthcare & Medicine**

* Disease Staging: Encoding the progression of illnesses (Stage I ➔ Stage II ➔ Stage III ➔ Stage IV) to predict patient survival rates.
* Pain Scales: Utilizing patient-reported pain levels (Mild ➔ Moderate ➔ Severe) to optimize triage priority in emergency rooms.

** Critical Rule for Implementation
Never use Ordinal Encoding for categories without a mathematical sequence (like Country or Color). Doing so forces a fake order (e.g., Spain = 1, France = 2, Germany = 3), which misleads distance-based machine learning models into thinking Germany is "greater than" Spain. Use One-Hot Encoding for those instead. [7, 8, 9] 


## 7 . Limitations and challenges

1. **Creation of a "False Order" (The biggest flaw)**: The algorithm assigns numbers ($0, 1, 2...$). Mathematical models will assume that category $2$ is "greater than" or "double" category $1$.

**Example**: If Red = 0, Blue = 1, and Green = 2, the computer will mathematically or graphically compute that Green is superior to Red, or that Green is closer to Blue than to Red. This introduces a false logical relationship.

2. **Sensitivity to Arbitrary Sorting**: Sorting alphabetically destroys the natural, inherent order of the data (e.g., ["Small", "Medium", "Large"] sorted alphabetically becomes Large (0), Medium (1), Small (2)).

3. **Distortion via Unknown Values**: Replacing all new or unseen categories with a single default value (like -1) creates a statistical distortion: all novelties are forced into the exact same group, which skews distance and correlation calculations.

## 8 · References
- Target and Categorical Encoding, Scikit-learn Documentation, https://scikit-learn.org/stable/modules/preprocessing.html#preprocessing-categorical-features
- All About Categorical Variable Encoding, Towards Data Science, https://towardsdatascience.com/all-about-categorical-variable-encoding-305f3361fd02/
- Guide to Ordinal Encoding in Machine Learning Pipelines, Analytics Vidhya, https://www.analyticsvidhya.com/blog/2020/03/one-hot-encoding-vs-label-encoding-using-scikit-learn/
- Preprocessing and Feature Engineering for Machine Learning, Sebastian Raschka (Lecture Notes), https://sebastianraschka.com/blog/2021/feature-engineering-part1.html
